# Stroke Prediction — Exploratory Data Analysis

**Research Question:** Can Generative AI solve the imbalanced data problem better than SMOTE?

**Dataset:** [Stroke Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset)  
**Target:** `stroke` (1 = had stroke, 0 = no stroke)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

DATA_PATH = Path('../data/healthcare-dataset-stroke-data.csv')

## 1. Load & Basic Overview

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Class Imbalance

In [ ]:
counts = df['stroke'].value_counts()
pct = df['stroke'].value_counts(normalize=True) * 100

print('Class distribution:')
print(pd.DataFrame({'count': counts, 'pct': pct.round(2)}))
print(f'\nImbalance ratio: {counts[0]/counts[1]:.1f}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['No Stroke (0)', 'Stroke (1)'], counts.values, color=['#2196F3', '#F44336'])
for i, (v, p) in enumerate(zip(counts.values, pct.values)):
    axes[0].text(i, v + 20, f'{v}\n({p:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=['No Stroke', 'Stroke'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=90, explode=(0, 0.05))
axes[1].set_title('Class Distribution (%)', fontsize=14)

plt.suptitle('Target Variable: Stroke', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['missing'] > 0]

if missing_df.empty:
    print('No missing values.')
else:
    print(missing_df)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(missing_df.index, missing_df['pct'], color='#FF9800')
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Feature')
    plt.tight_layout()
    plt.show()

## 4. Numerical Features

In [ ]:
num_features = ['age', 'avg_glucose_level', 'bmi']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, feat in enumerate(num_features):
    # Distribution by class
    ax = axes[0, i]
    for label, color, name in [(0, '#2196F3', 'No Stroke'), (1, '#F44336', 'Stroke')]:
        subset = df[df['stroke'] == label][feat].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(f'{feat} — Distribution by Class')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend()

    # Boxplot by class
    ax2 = axes[1, i]
    data_to_plot = [df[df['stroke'] == 0][feat].dropna(),
                    df[df['stroke'] == 1][feat].dropna()]
    bp = ax2.boxplot(data_to_plot, labels=['No Stroke', 'Stroke'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#2196F3')
    bp['boxes'][1].set_facecolor('#F44336')
    for patch in bp['boxes']:
        patch.set_alpha(0.7)
    ax2.set_title(f'{feat} — Boxplot by Class')

plt.suptitle('Numerical Features vs Stroke', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/numerical_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Categorical Features

In [ ]:
cat_features = ['gender', 'hypertension', 'heart_disease', 'ever_married',
                'work_type', 'Residence_type', 'smoking_status']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(cat_features):
    ax = axes[i]
    stroke_rate = df.groupby(feat)['stroke'].mean() * 100
    stroke_rate = stroke_rate.sort_values(ascending=False)

    bars = ax.bar(stroke_rate.index, stroke_rate.values, color='#E91E63', alpha=0.8)
    for bar, val in zip(bars, stroke_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'Stroke rate by {feat}', fontsize=11)
    ax.set_ylabel('Stroke %')
    ax.tick_params(axis='x', rotation=30)

axes[-1].set_visible(False)
plt.suptitle('Stroke Rate by Categorical Feature', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/categorical_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation with Target

In [ ]:
df_encoded = df.copy()

# Encode binary/ordinal categoricals for correlation
df_encoded['gender_enc'] = (df['gender'] == 'Male').astype(int)
df_encoded['married_enc'] = (df['ever_married'] == 'Yes').astype(int)
df_encoded['residence_enc'] = (df['Residence_type'] == 'Urban').astype(int)
df_encoded['smoking_enc'] = df['smoking_status'].map({
    'never smoked': 0, 'Unknown': 0, 'formerly smoked': 1, 'smokes': 2
})
df_encoded['work_enc'] = df['work_type'].map({
    'children': 0, 'Never_worked': 0, 'Govt_job': 1, 'Self-employed': 2, 'Private': 3
})

corr_features = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease',
                 'gender_enc', 'married_enc', 'residence_enc', 'smoking_enc', 'work_enc']

corr = df_encoded[corr_features + ['stroke']].corr()['stroke'].drop('stroke').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#F44336' if v > 0 else '#2196F3' for v in corr.values]
bars = ax.barh(corr.index, corr.values, color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Stroke (target)', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
for bar, val in zip(bars, corr.values):
    ax.text(val + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('../data/correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. EDA Summary

Key findings before modeling:

In [ ]:
stroke_df = df[df['stroke'] == 1]
no_stroke_df = df[df['stroke'] == 0]

print('=== EDA Summary ===')
print(f'Total samples:     {len(df):,}')
print(f'Stroke cases:      {len(stroke_df):,} ({len(stroke_df)/len(df)*100:.1f}%)')
print(f'No stroke cases:   {len(no_stroke_df):,} ({len(no_stroke_df)/len(df)*100:.1f}%)')
print(f'Imbalance ratio:   {len(no_stroke_df)/len(stroke_df):.1f}:1')
print()
print(f'Missing values:    {df.isnull().sum().sum()}')
print()
print('Mean age:')
print(f'  Stroke:     {stroke_df["age"].mean():.1f} years')
print(f'  No stroke:  {no_stroke_df["age"].mean():.1f} years')
print()
print('Mean avg_glucose_level:')
print(f'  Stroke:     {stroke_df["avg_glucose_level"].mean():.1f}')
print(f'  No stroke:  {no_stroke_df["avg_glucose_level"].mean():.1f}')
print()
print('Top correlated features (absolute):')
print(corr.abs().sort_values(ascending=False).head(5))